In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
import sys, os
from pathlib import Path

# Ensure project root is on sys.path and is the working directory
_root = Path().absolute()
# Navigate up until we find the rca_pipeline package
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Working directory: {os.getcwd()}")


In [ ]:
import pandas as pd
import re
import numpy as np
import html

In [ ]:
def clean_and_anonymize_data(df, text_col):    
    df[text_col] = df[text_col].fillna(' ').astype(str) # addded line to handle NaN values
    html_pattern = re.compile('<[^>]+>')
    df['Cleaned String'] = df[text_col].astype(str).str.replace(html_pattern, '', regex=True)
    df['Cleaned String'] = df['Cleaned String'].apply(html.unescape)
    df['Cleaned String'] = df['Cleaned String'].str.replace('&nbsp;', ' ', regex=False)
    df['Cleaned String'] = df['Cleaned String'].str.replace('\u00a0', ' ', regex=False)
    # name_pattern_dr = re.compile(r'(Dr\.\s*)[A-Z][a-z]+', re.IGNORECASE)
    # df['Cleaned String'] = df['Cleaned String'].str.replace(name_pattern_dr, r'\1[ANONYMIZED_NAME]', regex=True)
    # email_pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
    # df['Cleaned String'] = df['Cleaned String'].str.replace(email_pattern, '[ANONYMIZED_EMAIL]', regex=True)
    # phone_pattern = re.compile(r'(?:\d{3}|\(\d{3}\))[-.\s]?\d{3}[-.\s]?\d{4}')
    # df['Cleaned String'] = df['Cleaned String'].str.replace(phone_pattern, '[ANONYMIZED_PHONE]', regex=True)
    df['Cleaned String'] = df['Cleaned String'].str.strip()
    space_pattern = re.compile(r'\s+')
    df['Cleaned String'] = df['Cleaned String'].str.replace(space_pattern, ' ', regex=True)
    return df

def collapse_data(df, date_col=None, role_col=None, channel_col=None):
    """
    Collapses the data by 'Reference #', formats the conversation, and includes
    the Incident ID, Contact Reason columns, and the chat start date in Y-m-d format.
    """
    if date_col is None:
        for candidate in ['Date of Event', 'DATE_CREATED', 'Date Created', 'Created Date', 'Date']:
            if candidate in df.columns:
                date_col = candidate
                break
    if date_col is None:
        raise KeyError("No date column found. Expected one of: 'Date of Event', 'Date Created', 'Created Date', 'Date'.")
    if role_col is None:
        for candidate in ['Chat Event', 'Thread Entry Type', 'THREAD_ENTRY_TYPE', 'Event Type', 'Role']:
            if candidate in df.columns:
                role_col = candidate
                break
    if role_col is None:
        raise KeyError("No role column found. Expected one of: 'Chat Event', 'Thread Entry Type', 'Event Type', 'Role'.")
    if channel_col is None:
        for candidate in ['Channel ID', 'Channel', 'Channel Name', 'INCIDENT_CHANNEL']:
            if candidate in df.columns:
                channel_col = candidate
                break
    if channel_col is None:
        raise KeyError("No channel column found. Expected one of: 'Channel ID', 'Channel', 'Channel Name'.")
    # Ensure date column is a datetime object; invalid source values become NaT
    date_values = df[date_col].astype("string").str.replace("'", "", regex=False)
    df[date_col] = pd.to_datetime(date_values, errors="coerce")

    # Sort by 'Reference #' and then by date for temporal logic
    df_sorted = df.sort_values(by=['INCIDENT_REF', date_col])

    # Format the conversation line
    def format_line(row):
        channel = str(row[channel_col]).strip().lower()
        entry_type = str(row[role_col]).strip()
        entry_type_lower = entry_type.lower()
        content = row['Cleaned String']
        if entry_type_lower == 'rule response':
            role = 'Automated Email'
        elif entry_type_lower == 'note':
            role = 'Agent Notes'
        elif entry_type_lower == 'staff account':
            role = 'Agent'
        elif entry_type_lower in {'customer', 'customer proxy'}:
            role = 'Customer'
        elif channel == 'chat' and entry_type_lower == 'chat':
            return str(content)
        else:
            role = 'Customer'
        return f"{role}: {content}"

    df_sorted['Conversation Line'] = df_sorted.apply(format_line, axis=1)

    # Collapse the data by 'INCIDENT_REF'
    collapsed_df = df_sorted.groupby('INCIDENT_REF').agg(
        # Include metadata columns using 'first'
        Incident_ID=('INCIDENT_ID', 'first'),
        Channel=(channel_col, 'first'),
        # Contact_Reason_Level_1=('Contact reason Level 1', 'first'),
        # Contact_Reason_Level_2=('Contact reason Level 2', 'first'),
        # Aggregate the chat start date using 'min' (the earliest date)
        Chat_Start_Date=(date_col, 'min'),
        # Aggregate the conversation
        Conversation=('Conversation Line', lambda x: '\n'.join(x.tolist()))
    ).reset_index()

    # Format the Chat_Start_Date column to Y-m-d
    collapsed_df['Chat_Start_Date'] = collapsed_df['Chat_Start_Date'].dt.strftime('%Y-%m-%d')

    return collapsed_df

In [ ]:
df = pd.read_csv(r'data\Scopus\scopus 2026_verbatim.tsv',
    sep="\t",
    encoding="utf-8",
    engine="python",)

In [ ]:
df.shape

In [ ]:
df.head(15)

In [ ]:
df.columns

In [ ]:
print(df[df['INCIDENT_CHANNEL'] == 'Chat'].iloc[12]['VERBATIM'])

In [ ]:
df['INCIDENT_CHANNEL'].value_counts()

In [ ]:
df['THREAD_ENTRY_TYPE'].value_counts()

In [ ]:
df_cleaned = clean_and_anonymize_data(df.copy(), text_col='VERBATIM')

In [ ]:
df_cleaned.head()

In [ ]:
final_df = collapse_data(df_cleaned)

In [ ]:
final_df.shape

In [ ]:
final_df.head()

In [ ]:
import pandas as pd

# 1. Clean column names
final_df.columns = final_df.columns.str.strip()

# 2. Ensure Chat_Start_Date is datetime
final_df['Chat_Start_Date'] = pd.to_datetime(final_df['Chat_Start_Date'])

# 3. Create the summary by grouping on year and month directly
# We use 'n' to get the month name and 'Y' for the year
monthly_summary = final_df.groupby([final_df['Chat_Start_Date'].dt.year.rename('Year'), 
                                    final_df['Chat_Start_Date'].dt.month_name().rename('Month'),
                                    final_df['Chat_Start_Date'].dt.month.rename('Month_Num')]) \
                    .size().reset_index(name='Total Chats')

# 4. Sort by Year and Month Number (so it's Jan, Feb, Mar... not alphabetical)
monthly_summary = monthly_summary.sort_values(['Year', 'Month_Num'])

# 5. Final Table cleanup
final_table = monthly_summary[['Year', 'Month', 'Total Chats']]

# Display in a nice format
print(final_table.to_string(index=False))

In [ ]:
final_df.shape

In [ ]:
final_df.head() 

In [ ]:
from pathlib import Path

output_path = Path("data\Scopus\Scopus cleaned 2026.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_excel(output_path, index=False)

In [ ]:
# Reproducible sample of 200 incidents with conversation fields anonymised
sample_df = final_df.sample(n=min(200, len(final_df)), random_state=42).copy()
conversation_cols = [
    col for col in sample_df.columns
    if 'conversation' in col.lower()
]

if not conversation_cols:
    raise ValueError('No conversation columns found in final_df')

email_pattern = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')
url_pattern = re.compile(r'https?://\S+')
orcid_pattern = re.compile(r'\b\d{4}-\d{4}-\d{4}-\d{3}[\dX]\b', re.IGNORECASE)
phone_pattern = re.compile(r'(?<![\w-])(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?){2,4}\d{2,4}(?![\w-])')
greeting_name_pattern = re.compile(
    r'\b(Dear|Hi|Hello)\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3}(?=[,!.])'
)
speaker_name_pattern = re.compile(
    r'(?m)^(?!Customer:|Agent:|Agent Notes:|Automated Email:|System:)'
    r'[A-Z][A-Za-z-]+(?:\s+[A-Za-z-]+){1,3}:'
)

def anonymise_conversation(value):
    text = str(value)
    text = email_pattern.sub('[ANONYMIZED_EMAIL]', text)
    text = orcid_pattern.sub('[ANONYMIZED_ORCID]', text)
    text = url_pattern.sub('[ANONYMIZED_URL]', text)

    def replace_phone(match):
        value = match.group(0)
        digits = re.sub(r'\D', '', value)
        # Do not redact ISO-style dates as phone numbers.
        if re.fullmatch(r'\d{4}[-/]\d{1,2}[-/]\d{1,2}', value.strip()):
            return value
        return '[ANONYMIZED_PHONE]' if len(digits) >= 8 else value

    text = phone_pattern.sub(replace_phone, text)
    text = greeting_name_pattern.sub(r'\1 [ANONYMIZED_NAME]', text)
    text = speaker_name_pattern.sub('[ANONYMIZED_NAME]:', text)
    return text

for col in conversation_cols:
    sample_df[col] = sample_df[col].map(anonymise_conversation)

sample_df.to_csv('data/scopus content/scopus content_2026_threads_collapsed_sample_200_anonymized.csv', index=False, encoding='utf-8')
print(f'Extracted {len(sample_df)} incidents')
print(f'Anonymised columns: {conversation_cols}')
sample_df.head()

In [ ]:
meta_df = pd.read_csv(r'data\Scopus\scopus 2026_metadata.tsv',
    sep="\t",
    encoding="utf-8",
    engine="python",)

In [ ]:
meta_df.shape

In [ ]:
merged_df = final_df.merge(
    meta_df,
    left_on="INCIDENT_REF",
    right_on="incident_reference",
    how="outer",
    indicator=True,
    validate="one_to_one",
)
merged_df.head()

In [ ]:
merged_df.shape

In [ ]:
merged_df.to_excel(output_path, index=False)